# NB16 — CPU stage: bootstrap CIs, validity fix, and external-benchmark preparation

Everything here runs **without a GPU**, so it must not be run in a GPU session (that would burn the
accelerator quota on CPU work). Requires **Internet ON**.

Stages:
1. Convert the NB13 generations to parquet with a composite `gen_id`
2. Re-define polish validity per level (the old jaccard<0.85 rule was wrong for low levels)
3. **Paired bootstrap confidence intervals** on the NB15 stress results — the headline numbers
4. Check whether the AraGenEval / ARATECT data is obtainable post-competition
5. Download and inspect Ar-APT (public external benchmark)
6. Compute Vstat for the Ar-APT articles with the saved train-fit scalers
7. Chunk the Ar-APT articles to K=9 (CPU work, kept out of the GPU session on purpose)

Outputs go to /kaggle/working; save them as a dataset and point NB17 at them.

## 1 · Config

In [1]:
import os, re, io, gzip, json, time, pickle, zipfile, subprocess, numpy as np, pandas as pd

# --- inputs you must set ---
P_DATASET  = "/kaggle/input/datasets/bahaaqassem/aig-and-humang-dataset/dataset.parquet"
P_GENS     = "/kaggle/input/datasets/bahaaqassem/stress-test-dataset/stress_generations.parquet"
P_STRESS   = "/kaggle/input/notebooks/bahaaqassem/nb15-stress-evaluation/nb15_stress_results.parquet"
P_SCALER5  = "/kaggle/input/datasets/bahaaqassem/scalers/scaler.pkl"
P_SCALER11 = "/kaggle/input/datasets/bahaaqassem/scalers/scaler11.pkl"

# --- knobs ---
N_BOOT     = 2000                 # bootstrap resamples
ARAPT_REPO = "https://github.com/Saleh-Almohaimeed/Ar-APT"
ARAPT_MAX  = 3000                 # cap on Ar-APT articles prepared (controls NB17 GPU time)
ARAPT_POLISH_MODELS = ["claude-4-sonnet", "GPT 4", "GPT4", "LLAMA", "DS", "QWEN M"]
ARAPT_PER_CELL = 100      # articles per (polishing model × level)
MODEL_ID   = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
K_CHUNKS, MAX_CT, STRIDE = 9, 510, 460          # frozen training contract — do not change
STAT_COLS  = ["burstiness","ttr","quote_ratio","function_word_ratio","compressibility"]
SEED = 42
np.random.seed(SEED)
print("[1/7] config loaded", flush=True)
print(f"      N_BOOT={N_BOOT} | ARAPT_MAX={ARAPT_MAX} | K={K_CHUNKS}", flush=True)

[1/7] config loaded
      N_BOOT=2000 | ARAPT_MAX=3000 | K=9


## 2 · Generations → parquet with gen_id, and validity redefined

In [2]:
G = pd.read_parquet(P_GENS)
print(f"[2/7] loaded parquet | columns: {list(G.columns)}", flush=True)
G["level"] = G["level"].astype("Float64")
if "gen_id" not in G.columns:
    G["gen_id"] = (G.article_id + "__" + G.model + "__" + G.task +
                   G.level.map(lambda x: f"__L{int(x)}" if pd.notna(x) else ""))
assert G.gen_id.is_unique
if "src_words" not in G.columns: G["src_words"] = df_src_words_placeholder  # see note below
if "out_words" not in G.columns: G["out_words"] = G.text.str.split().str.len()
print(f"[2/7] generations {len(G)} | tasks {G.task.value_counts().to_dict()}", flush=True)

df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
src = df["text"].astype(str)

if {"cosine","jaccard"}.issubset(G.columns):
    print("[2/7] cosine/jaccard already present — skipping recomputation", flush=True)
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    cos, jac = [], []
    for k, r in enumerate(G.itertuples(), 1):
        a, b = src[r.article_id], r.text
        try:
            V = TfidfVectorizer().fit_transform([a, b]); c = float(cosine_similarity(V[0], V[1])[0,0])
        except Exception: c = float("nan")
        A, B = set(a.split()), set(b.split())
        cos.append(c); jac.append(len(A & B)/max(len(A | B), 1))
        if k % 200 == 0: print(f"[2/7]   scored {k}/{len(G)}", flush=True)
    G["cosine"], G["jaccard"] = cos, jac
G["len_ratio"] = G.out_words / G.src_words.clip(lower=1)

# OLD rule (kept for transparency) vs NEW rule
G["valid_old"] = (G.cosine > 0.5) & (G.jaccard < 0.85) & G.len_ratio.between(0.6, 1.6)
# NEW: garbled = low cosine or wild length; no-op = essentially unchanged text.
# A 10% polish SHOULD leave a high jaccard, so the ceiling is a no-op test, not a level test.
G["garbled"] = (G.cosine <= 0.5) | (~G.len_ratio.between(0.6, 1.6))
G["noop"]    = G.jaccard > 0.98
G["valid"]   = ~(G.garbled | G.noop)

print(f"\n[2/7] valid_old {int(G.valid_old.sum())}/{len(G)} ({100*G.valid_old.mean():.1f}%)", flush=True)
print(f"[2/7] valid_new {int(G.valid.sum())}/{len(G)} ({100*G.valid.mean():.1f}%) "
      f"| garbled {int(G.garbled.sum())} | no-op {int(G.noop.sum())}", flush=True)
print("\n[2/7] polish levels under the NEW rule (monotone jaccard = protocol worked):", flush=True)
print(G[G.task=="polish"].groupby("level").agg(
      n=("valid","size"), cosine=("cosine","mean"), jaccard=("jaccard","mean"),
      valid_old=("valid_old","mean"), valid_new=("valid","mean")).round(3).to_string(), flush=True)
print("\n[2/7] no-op rate by task x model (exposes models that ignored the instruction):", flush=True)
print(G.groupby(["task","model"])["noop"].mean().round(3).to_string(), flush=True)

G.to_parquet("/kaggle/working/stress_generations.parquet", index=False)
G.drop(columns=["text"]).to_parquet("/kaggle/working/stress_generations_meta.parquet", index=False)
print("\n[2/7] saved stress_generations.parquet (+ meta)", flush=True)

[2/7] loaded parquet | columns: ['article_id', 'model', 'provider', 'task', 'level', 'src_words', 'out_words', 'tok_in', 'tok_out', 'cost', 'secs', 'text', 'gen_id']
[2/7] generations 820 | tasks {'humanize': 410, 'polish': 410}
[2/7]   scored 200/820
[2/7]   scored 400/820
[2/7]   scored 600/820
[2/7]   scored 800/820

[2/7] valid_old 652/820 (79.5%)
[2/7] valid_new 748/820 (91.2%) | garbled 9 | no-op 63

[2/7] polish levels under the NEW rule (monotone jaccard = protocol worked):
         n  cosine  jaccard  valid_old  valid_new
level                                            
10.0   105   0.957    0.849      0.381      0.771
25.0   105   0.921    0.749      0.552      0.790
50.0   100   0.885    0.654      0.760      0.860
75.0   100   0.838    0.557      0.820      0.940

[2/7] no-op rate by task x model (exposes models that ignored the instruction):
task      model   
humanize  claude      0.000
          deepseek    0.000
          gemini      0.000
          gpt         0.000
 

## 3 · Paired bootstrap CIs on the NB15 stress results

The two detectors are scored on the SAME articles, so the resampling is **paired**: we resample
articles with replacement and recompute both detectors on the resample, then take percentiles of the
**difference**. That gives a CI on the gap, which is the claim — not on each rate separately.

In [3]:
S = pd.read_parquet(P_STRESS)
print(f"[3/7] loaded stress results: {len(S)} rows | cols {[c for c in S.columns if c.startswith('p_')]}", flush=True)

def boot_ci(mask, col_h, col_n, positive_is, B=N_BOOT):
    sub = S[mask]
    if len(sub) == 0: return None
    ph, pn = sub[col_h].to_numpy(), sub[col_n].to_numpy()
    idx = np.random.randint(0, len(sub), size=(B, len(sub)))
    rh = ( (ph[idx] >= .5).mean(1) )*100
    rn = ( (pn[idx] >= .5).mean(1) )*100
    d  = rh - rn
    q  = lambda a: (np.percentile(a, 2.5), np.percentile(a, 97.5))
    return {"n": len(sub),
            "hyb": round(float((ph>=.5).mean()*100),2), "hyb_lo": round(q(rh)[0],2), "hyb_hi": round(q(rh)[1],2),
            "neu": round(float((pn>=.5).mean()*100),2), "neu_lo": round(q(rn)[0],2), "neu_hi": round(q(rn)[1],2),
            "gap": round(float((ph>=.5).mean()*100 - (pn>=.5).mean()*100),2),
            "gap_lo": round(q(d)[0],2), "gap_hi": round(q(d)[1],2),
            "sig": "YES" if (q(d)[0] > 0 or q(d)[1] < 0) else "no"}

print("\n[3/7] A · POLISHING — rate = FPR (lower better). gap = hybrid − neural", flush=True)
out=[]
for lvl in sorted(S[S.task=="polish"].level.dropna().unique()):
    r = boot_ci((S.task=="polish") & (S.level==lvl), "p_hybrid_gen", "p_neural_gen", "FPR")
    r = {"level": str(int(lvl)), **r}; out.append(r)
r = boot_ci(S.task=="polish", "p_hybrid_gen", "p_neural_gen", "FPR"); out.append({"level":"ALL", **r})
A = pd.DataFrame(out); print(A.to_string(index=False), flush=True)

print("\n[3/7] B · HUMANIZING — rate = TPR (higher better). gap = hybrid − neural", flush=True)
out=[]
r = boot_ci(S.task=="humanize", "p_hybrid_gen", "p_neural_gen", "TPR"); out.append({"scope":"ALL", **r})
for m in sorted(S[S.task=="humanize"].model.unique()):
    r = boot_ci((S.task=="humanize") & (S.model==m), "p_hybrid_gen", "p_neural_gen", "TPR")
    out.append({"scope": m, **r})
Bt = pd.DataFrame(out); print(Bt.to_string(index=False), flush=True)

print("\n[3/7] C · UNTRANSFORMED baseline on the same articles (sanity: gap should be ~0)", flush=True)
out=[]
for tk in ("polish","humanize"):
    r = boot_ci(S.task==tk, "p_hybrid_base", "p_neural_base", "-"); out.append({"task":tk, **r})
C = pd.DataFrame(out); print(C.to_string(index=False), flush=True)

A.to_parquet("/kaggle/working/boot_polish.parquet", index=False)
Bt.to_parquet("/kaggle/working/boot_humanize.parquet", index=False)
C.to_parquet("/kaggle/working/boot_baseline.parquet", index=False)
print("\n[3/7] sig=YES means the 95% CI of the gap excludes zero -> the difference is real, not noise", flush=True)
print("[3/7] saved boot_*.parquet", flush=True)

[3/7] loaded stress results: 820 rows | cols ['p_hybrid_base', 'p_hybrid_gen', 'p_neural_base', 'p_neural_gen']

[3/7] A · POLISHING — rate = FPR (lower better). gap = hybrid − neural
level   n   hyb  hyb_lo  hyb_hi   neu  neu_lo  neu_hi   gap  gap_lo  gap_hi sig
   10 105  2.86    0.00    6.67  3.81    0.95    7.62 -0.95   -2.86    0.00  no
   25 105  1.90    0.00    4.76  6.67    1.90   11.43 -4.76   -9.52   -0.95 YES
   50 100 10.00    4.00   16.00 12.00    6.00   18.00 -2.00   -6.00    2.00  no
   75 100 21.00   13.98   29.00 25.00   17.00   34.00 -4.00  -10.00    1.00  no
  ALL 410  8.78    6.10   11.46 11.71    8.78   15.12 -2.93   -5.12   -0.98 YES

[3/7] B · HUMANIZING — rate = TPR (higher better). gap = hybrid − neural
   scope   n    hyb  hyb_lo  hyb_hi    neu  neu_lo  neu_hi    gap  gap_lo  gap_hi sig
     ALL 410  94.15   91.71   96.34  97.07   95.37   98.54  -2.93   -4.63   -1.46 YES
  claude  82  76.83   68.29   85.37  89.02   81.71   95.12 -12.20  -19.51   -6.10 YES
deep

## 4 · Is the AraGenEval / ARATECT data obtainable?

In [4]:
!pip install -q requests
import requests
def probe(url, tag):
    try:
        r = requests.get(url, timeout=25)
        print(f"[4/7]   {tag}: HTTP {r.status_code} ({len(r.content)} bytes)", flush=True)
        return r
    except Exception as e:
        print(f"[4/7]   {tag}: FAILED {e}", flush=True); return None

print("[4/7] probing AraGenEval sources", flush=True)
r = probe("https://api.github.com/repos/ezzini/AraGenEval/contents/", "repo root")
if r is not None and r.status_code == 200:
    try:
        for it in r.json():
            print(f"[4/7]     - {it['type']:4s} {it['name']} ({it.get('size',0)} b)", flush=True)
    except Exception as e:
        print(f"[4/7]     could not parse listing: {e}", flush=True)
probe("https://api.github.com/repos/ezzini/AraGenEval/contents/data", "data/ dir")
print("[4/7] NOTE: the shared task distributed data through Codabench, which usually requires", flush=True)
print("[4/7]       registration. If nothing above lists usable files, we drop the external", flush=True)
print("[4/7]       AraGenEval test and instead REIMPLEMENT their architecture on our corpus.", flush=True)

[4/7] probing AraGenEval sources
[4/7]   repo root: HTTP 200 (5167 bytes)
[4/7]     - file LICENSE (1061 b)
[4/7]     - file PAPER.md (3182 b)
[4/7]     - file _config.yml (309 b)
[4/7]     - dir  _layouts (0 b)
[4/7]     - file googlec4184caf97087de3.html (53 b)
[4/7]     - file guidelines.md (5170 b)
[4/7]     - file index.md (8756 b)
[4/7]   data/ dir: HTTP 404 (127 bytes)
[4/7] NOTE: the shared task distributed data through Codabench, which usually requires
[4/7]       registration. If nothing above lists usable files, we drop the external
[4/7]       AraGenEval test and instead REIMPLEMENT their architecture on our corpus.


## 5 · Download and inspect Ar-APT

In [5]:
ARAPT_DIR = "/kaggle/working/Ar-APT"
if not os.path.exists(ARAPT_DIR):
    print("[5/7] cloning Ar-APT ...", flush=True)
    rc = subprocess.run(["git","clone","--depth","1",ARAPT_REPO+".git",ARAPT_DIR],
                        capture_output=True, text=True)
    print(f"[5/7]   git rc={rc.returncode} {rc.stderr[:200]}", flush=True)
    if rc.returncode != 0:
        print("[5/7]   falling back to zip download", flush=True)
        for br in ("main","master"):
            try:
                z = requests.get(f"https://codeload.github.com/Saleh-Almohaimeed/Ar-APT/zip/refs/heads/{br}", timeout=120)
                if z.status_code == 200:
                    zipfile.ZipFile(io.BytesIO(z.content)).extractall("/kaggle/working/")
                    for d in os.listdir("/kaggle/working"):
                        if d.startswith("Ar-APT"): ARAPT_DIR = "/kaggle/working/"+d
                    print(f"[5/7]   extracted -> {ARAPT_DIR}", flush=True); break
            except Exception as e:
                print(f"[5/7]   {br} failed: {e}", flush=True)
else:
    print("[5/7] Ar-APT already present", flush=True)

print("[5/7] repository tree:", flush=True)
found = []
for root, dirs, files in os.walk(ARAPT_DIR):
    dirs[:] = [d for d in dirs if d != ".git"]
    for f in sorted(files):
        p = os.path.join(root, f)
        rel = os.path.relpath(p, ARAPT_DIR)
        sz = os.path.getsize(p)
        print(f"[5/7]   {rel}  ({sz/1024:.0f} KB)", flush=True)
        if f.lower().endswith((".csv",".tsv",".json",".jsonl",".parquet",".xlsx")):
            found.append(p)
print(f"[5/7] {len(found)} loadable data files", flush=True)

[5/7] cloning Ar-APT ...
[5/7]   git rc=0 Cloning into '/kaggle/working/Ar-APT'...

[5/7] repository tree:
[5/7]   .DS_Store  (6 KB)
[5/7]   AI 10 LLMs generated Articles.json  (1032 KB)
[5/7]   README.md  (0 KB)
[5/7]   Ar-APT/.DS_Store  (14 KB)
[5/7]   Ar-APT/AI Polished Texts/.DS_Store  (6 KB)
[5/7]   Ar-APT/AI Polished Texts/DS 10%.json  (1809 KB)
[5/7]   Ar-APT/AI Polished Texts/DS 25%.json  (1798 KB)
[5/7]   Ar-APT/AI Polished Texts/DS 50%.json  (1788 KB)
[5/7]   Ar-APT/AI Polished Texts/DS 75%.json  (1803 KB)
[5/7]   Ar-APT/AI Polished Texts/GEMMA 10%.json  (1753 KB)
[5/7]   Ar-APT/AI Polished Texts/GEMMA 25%.json  (1709 KB)
[5/7]   Ar-APT/AI Polished Texts/GEMMA 50%.json  (1733 KB)
[5/7]   Ar-APT/AI Polished Texts/GEMMA 75%.json  (1798 KB)
[5/7]   Ar-APT/AI Polished Texts/GPT 3.5 10%.json  (1778 KB)
[5/7]   Ar-APT/AI Polished Texts/GPT 3.5 25%.json  (1785 KB)
[5/7]   Ar-APT/AI Polished Texts/GPT 3.5 50%.json  (1773 KB)
[5/7]   Ar-APT/AI Polished Texts/GPT 3.5 75%.json  (1772 KB

## 6 · Parse Ar-APT into a single table

In [6]:
import glob
AR_ROOT = ARAPT_DIR
parts = []

# --- 1) human UNPOLISHED control (the .txt files) ---
txts = sorted(glob.glob(os.path.join(AR_ROOT, "**", "Human-Written-Text", "*.txt"), recursive=True))
print(f"[6/7] human .txt files: {len(txts)}", flush=True)
for p in txts:
    try: t = open(p, encoding="utf-8", errors="ignore").read().strip()
    except Exception: continue
    if len(t.split()) < 50: continue
    dom = os.path.basename(p).split("-")[0]
    parts.append({"text": t, "label": 0, "kind": "human_orig",
                  "polish_model": None, "level": None, "domain": dom})
print(f"[6/7]   loaded {len(parts)} unpolished human articles", flush=True)

# --- 2) human POLISHED (Polish_text column, NOT 'original') ---
pol_files = sorted(glob.glob(os.path.join(AR_ROOT, "**", "AI Polished Texts", "*.json"), recursive=True))
print(f"[6/7] polished files: {len(pol_files)}", flush=True)
n_pol = 0
for p in pol_files:
    stem = os.path.splitext(os.path.basename(p))[0].strip()
    m = re.match(r'^(.*?)\s*(\d+)\s*%\s*$', stem)
    if not m: print(f"[6/7]   ? cannot parse name: {stem}", flush=True); continue
    mdl, lvl = m.group(1).strip(), int(m.group(2))
    if ARAPT_POLISH_MODELS and mdl not in ARAPT_POLISH_MODELS: continue
    try: d = pd.read_json(p)
    except Exception as e: print(f"[6/7]   ! {stem}: {e}", flush=True); continue
    if "Polish_text" not in d.columns:
        print(f"[6/7]   ! {stem}: no Polish_text column", flush=True); continue
    if "Polish_Percentage" in d.columns:
        try:
            v = float(pd.to_numeric(d["Polish_Percentage"], errors="coerce").dropna().mode().iloc[0])
            lvl = int(round(v*100)) if v <= 1.0 else int(round(v))   # stored as a fraction
        except Exception: pass
    d = d.head(ARAPT_PER_CELL)
    for _, r in d.iterrows():
        t = str(r["Polish_text"]).strip()
        if len(t.split()) < 50: continue
        parts.append({"text": t, "label": 0, "kind": "human_polished",
                      "polish_model": mdl, "level": lvl,
                      "domain": r.get("domain", None)})
        n_pol += 1
    print(f"[6/7]   {mdl:18s} L{lvl:<3d} -> {len(d)} rows", flush=True)
print(f"[6/7]   loaded {n_pol} polished articles", flush=True)

# --- 3) the AI-generated set (structure unknown -> inspect then branch) ---
aip = os.path.join(AR_ROOT, "AI 10 LLMs generated Articles.json")
if os.path.exists(aip):
    d = pd.read_json(aip)
    print(f"[6/7] AI file {d.shape} cols={list(d.columns)}", flush=True)
    lab_col = "AI/Human" if "AI/Human" in d.columns else None
    if lab_col is not None:
        avg = d[lab_col].astype(str).str.len().mean()
        print(f"[6/7]   '{lab_col}' avg len {avg:.0f} | sample {d[lab_col].astype(str).head(3).tolist()}", flush=True)
        if avg > 200:                       # the column holds AI TEXT
            for _, r in d.iterrows():
                t = str(r[lab_col]).strip()
                if len(t.split()) >= 50:
                    parts.append({"text": t, "label": 1, "kind": "ai_generated",
                                  "polish_model": None, "level": None,
                                  "domain": r.get("domain", None)})
        else:                               # the column holds a LABEL
            print(f"[6/7]   label values: {d[lab_col].value_counts().to_dict()}", flush=True)
            for _, r in d.iterrows():
                t = str(r["original"]).strip()
                lb = 1 if str(r[lab_col]).strip().upper().startswith("AI") else 0
                if len(t.split()) >= 50:
                    parts.append({"text": t, "label": lb,
                                  "kind": "ai_generated" if lb == 1 else "human_orig2",
                                  "polish_model": None, "level": None,
                                  "domain": r.get("domain", None)})
else:
    print("[6/7] !! AI-generated file not found", flush=True)

AR = pd.DataFrame(parts)
if len(AR) > ARAPT_MAX:
    AR = AR.groupby("kind", group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, int(ARAPT_MAX*len(g)/len(AR)))), random_state=SEED))
AR = AR.reset_index(drop=True)
AR["arapt_id"] = [f"ARAPT_{i:05d}" for i in range(len(AR))]
print(f"\n[6/7] Ar-APT prepared: {len(AR)} articles | median {int(AR.text.str.split().str.len().median())} words", flush=True)
print(f"[6/7] by kind: {AR.kind.value_counts().to_dict()}", flush=True)
print(f"[6/7] by label: {AR.label.value_counts().to_dict()}", flush=True)
print("[6/7] polished cells (model x level):", flush=True)
pp = AR[AR.kind=="human_polished"]
if len(pp): print(pp.groupby(["polish_model","level"]).size().to_string(), flush=True)
AR.to_parquet("/kaggle/working/arapt_prepared.parquet", index=False)
print("[6/7] saved arapt_prepared.parquet", flush=True)

[6/7] human .txt files: 400
[6/7]   loaded 400 unpolished human articles
[6/7] polished files: 40
[6/7]   DS                 L10  -> 100 rows
[6/7]   DS                 L25  -> 100 rows
[6/7]   DS                 L50  -> 100 rows
[6/7]   DS                 L75  -> 100 rows
[6/7]   GPT 4              L10  -> 100 rows
[6/7]   GPT 4              L25  -> 100 rows
[6/7]   GPT4               L50  -> 100 rows
[6/7]   GPT4               L75  -> 100 rows
[6/7]   LLAMA              L10  -> 100 rows
[6/7]   LLAMA              L25  -> 100 rows
[6/7]   LLAMA              L50  -> 100 rows
[6/7]   LLAMA              L75  -> 100 rows
[6/7]   QWEN M             L10  -> 100 rows
[6/7]   QWEN M             L25  -> 100 rows
[6/7]   QWEN M             L50  -> 100 rows
[6/7]   QWEN M             L75  -> 100 rows
[6/7]   claude-4-sonnet    L10  -> 100 rows
[6/7]   claude-4-sonnet    L25  -> 100 rows
[6/7]   claude-4-sonnet    L50  -> 100 rows
[6/7]   claude-4-sonnet    L75  -> 100 rows
[6/7]   loaded 1936 po

## 7 · Vstat + K=9 chunking for Ar-APT (all CPU, keeps NB17's GPU time down)

In [7]:
if AR is not None:
    _SENT = re.compile(r'[.!?\u061f\u0964\n]+')
    def split_sentences(t): return [s.strip() for s in _SENT.split(t) if s.strip()]
    def tokenize_ws(t):     return [x for x in re.split(r'\s+', t.strip()) if x]
    AR_DIAC = re.compile(r'[\u064b-\u0652\u0670\u0640]')
    def strip_diac(t):      return AR_DIAC.sub('', t)
    def burstiness(text):
        L = np.array([len(tokenize_ws(s)) for s in split_sentences(text)], dtype=float)
        if L.size < 2: return float('nan')
        mu, sg = L.mean(), L.std()
        return 0.0 if (sg+mu)==0 else float((sg-mu)/(sg+mu))
    def mattr(tk, w=100):
        if len(tk) < w: return len(set(tk))/len(tk) if tk else float('nan')
        return float(np.mean([len(set(tk[i:i+w]))/w for i in range(len(tk)-w+1)]))
    def ttr_feat(t): return mattr(tokenize_ws(t), 100)
    QP = [('\u00ab','\u00bb'),('\u201c','\u201d'),('"','"'),("'","'")]
    def quote_ratio(text):
        tk = tokenize_ws(text)
        if not tk: return float('nan')
        ins = 0
        for op, cl in QP:
            if op == cl:
                ps = text.split(op)
                for k in range(1, len(ps), 2): ins += len(tokenize_ws(ps[k]))
            else:
                for m in re.finditer(re.escape(op)+r'(.*?)'+re.escape(cl), text, flags=re.S):
                    ins += len(tokenize_ws(m.group(1)))
        return float(min(ins, len(tk)))/len(tk)
    FW = set(['\u0641\u064a','\u0645\u0646','\u0625\u0644\u0649','\u0639\u0644\u0649','\u0639\u0646','\u0645\u0639',
        '\u0628\u064a\u0646','\u0639\u0646\u062f','\u0644\u062f\u0649','\u062e\u0644\u0627\u0644','\u0628\u0639\u062f',
        '\u0642\u0628\u0644','\u0645\u0646\u0630','\u062d\u062a\u0649','\u0625\u0646','\u0623\u0646','\u0623\u0646\u0647',
        '\u0625\u0646\u0647','\u0627\u0644\u0630\u064a','\u0627\u0644\u062a\u064a','\u0647\u0630\u0627','\u0647\u0630\u0647',
        '\u0630\u0644\u0643','\u062a\u0644\u0643','\u0647\u0648','\u0647\u064a','\u0647\u0645','\u0647\u0646','\u0646\u062d\u0646',
        '\u0623\u0646\u0627','\u0643\u0644','\u0628\u0639\u0636','\u063a\u064a\u0631','\u0644\u0627','\u0645\u0627','\u0644\u0645',
        '\u0644\u0646','\u0642\u062f','\u0644\u0642\u062f','\u0623\u064a','\u0623\u064a\u0636\u0627','\u0643\u0645\u0627',
        '\u0644\u0643\u0646','\u0623\u0648','\u062b\u0645','\u0628\u0644','\u0625\u0630','\u0625\u0630\u0627','\u0644\u0623\u0646','\u062d\u064a\u062b'])
    def function_word_ratio(text):
        tk = [strip_diac(x) for x in tokenize_ws(text)]
        if not tk: return float('nan')
        return float(sum(1 for x in tk if x in FW))/len(tk)
    def compressibility(text):
        raw = text.encode('utf-8')
        if len(raw) < 200: return float('nan')
        return float(len(gzip.compress(raw, compresslevel=6)))/float(len(raw))
    FEAT = {"burstiness":burstiness,"ttr":ttr_feat,"quote_ratio":quote_ratio,
            "function_word_ratio":function_word_ratio,"compressibility":compressibility}

    s5, s11 = pickle.load(open(P_SCALER5,"rb")), pickle.load(open(P_SCALER11,"rb"))
    PARAM = {}
    for srcp in (s5, s11):
        order=list(srcp["feature_order"]); sc=srcp["scaler"]; med=np.asarray(srcp["train_median"])
        for f in STAT_COLS:
            if f in order and f not in PARAM:
                i=order.index(f); PARAM[f]=(float(sc.mean_[i]), float(sc.scale_[i]), float(med[i]))
    print(f"[7/7] scaler params loaded for {list(PARAM)}", flush=True)

    X = np.empty((len(AR), len(STAT_COLS)), np.float32); t0=time.time()
    for i, txt in enumerate(AR.text.tolist()):
        for j, f in enumerate(STAT_COLS):
            try: v = FEAT[f](str(txt))
            except Exception: v = float('nan')
            m, s, med = PARAM[f]
            if v is None or (isinstance(v,float) and np.isnan(v)): v = med
            X[i,j] = (v - m)/s
        if (i+1)%300==0: print(f"[7/7]   featurised {i+1}/{len(AR)} ({time.time()-t0:.0f}s)", flush=True)
    pd.DataFrame(X, columns=STAT_COLS).assign(arapt_id=AR.arapt_id.values) \
      .to_parquet("/kaggle/working/arapt_vstat.parquet", index=False)
    print(f"[7/7] Vstat {X.shape} saved", flush=True)
    for j,f in enumerate(STAT_COLS):
        print(f"[7/7]   {f:22s} mean {X[:,j].mean():+.3f}", flush=True)

    print("[7/7] feature means BY KIND (isolates domain shift from polishing):", flush=True)
    FD = pd.DataFrame(X, columns=STAT_COLS); FD["kind"] = AR.kind.values; FD["level"] = AR.level.values
    print(FD.groupby("kind")[STAT_COLS].mean().round(3).to_string(), flush=True)
    pol_only = FD[FD.kind=="human_polished"]
    if len(pol_only): print(pol_only.groupby("level")[STAT_COLS].mean().round(3).to_string(), flush=True)
    !pip install -q transformers
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
    CL = MAX_CT + 2
    CH = np.full((len(AR), K_CHUNKS, CL), PAD, np.int32); NC = np.zeros(len(AR), np.int8)
    t0=time.time()
    for i, t in enumerate(AR.text.tolist()):
        ids = tok(str(t), add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
        wins = [ids[k:k+MAX_CT] for k in range(0,len(ids),STRIDE)][:K_CHUNKS] or [ids[:MAX_CT]]
        NC[i]=len(wins)
        for j,w in enumerate(wins): CH[i,j]=[CLS]+w+[SEP]+[PAD]*(CL-2-len(w))
        if (i+1)%300==0: print(f"[7/7]   chunked {i+1}/{len(AR)} ({time.time()-t0:.0f}s)", flush=True)
    pd.DataFrame({"arapt_id": AR.arapt_id.values,
                  "n_chunks": NC.astype("int16"),
                  "chunk_shape": [list(CH.shape[1:])]*len(AR),
                  "chunks": [r.reshape(-1).astype("int32").tolist() for r in CH]}) \
      .to_parquet("/kaggle/working/arapt_chunks_K9.parquet", index=False)
    print(f"[7/7] chunks {CH.shape} saved | mean chunks/article {NC.mean():.2f}", flush=True)
    print("\n[7/7] READY FOR NB17: arapt_prepared.parquet, arapt_vstat.parquet, arapt_chunks_K9.parquet", flush=True)
else:
    print("[7/7] skipped — Ar-APT not parsed", flush=True)
print("\n[DONE] save /kaggle/working as a dataset and point NB17 at it", flush=True)

[7/7] scaler params loaded for ['burstiness', 'ttr', 'quote_ratio', 'function_word_ratio', 'compressibility']


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[7/7]   featurised 300/2736 (0s)
[7/7]   featurised 600/2736 (1s)
[7/7]   featurised 900/2736 (1s)
[7/7]   featurised 1200/2736 (1s)
[7/7]   featurised 1500/2736 (2s)
[7/7]   featurised 1800/2736 (2s)
[7/7]   featurised 2100/2736 (2s)
[7/7]   featurised 2400/2736 (3s)
[7/7]   featurised 2700/2736 (3s)
[7/7] Vstat (2736, 5) saved
[7/7]   burstiness             mean +0.129
[7/7]   ttr                    mean -0.559
[7/7]   quote_ratio            mean -0.501
[7/7]   function_word_ratio    mean -0.553
[7/7]   compressibility        mean +3.219
[7/7] feature means BY KIND (isolates domain shift from polishing):
                burstiness    ttr  quote_ratio  function_word_ratio  compressibility
kind                                                                                
ai_generated        -0.580  0.005       -0.710                0.309            2.674
human_orig           0.425 -1.407       -0.483               -0.601            2.834
human_polished       0.215 -0.501       -0.462

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[7/7]   chunked 300/2736 (0s)
[7/7]   chunked 600/2736 (1s)
[7/7]   chunked 900/2736 (1s)
[7/7]   chunked 1200/2736 (1s)
[7/7]   chunked 1500/2736 (1s)
[7/7]   chunked 1800/2736 (2s)
[7/7]   chunked 2100/2736 (2s)
[7/7]   chunked 2400/2736 (2s)
[7/7]   chunked 2700/2736 (3s)
[7/7] chunks (2736, 9, 512) saved | mean chunks/article 1.09

[7/7] READY FOR NB17: arapt_prepared.parquet, arapt_vstat.parquet, arapt_chunks_K9.parquet

[DONE] save /kaggle/working as a dataset and point NB17 at it
